# FYP 2: Keypoint Extraction — Nandana et al. 2026 Dataset
**Author:** Sofia Binti Saiful Rizal  
**Dataset:** Nandana et al. (2026) — Upper Limb Stroke Rehabilitation Exercises  

Runs MediaPipe BlazePose and YOLOv8n-Pose on every video in the dataset and saves
per-frame landmark coordinates to CSV, mirroring the original folder structure:

```
An_upper_limb_.../
  MediaPipe_CSV/
    1_Lifting an Object/Complete/Test/  ← *_mp.csv
    ...
  YOLO_CSV/
    1_Lifting an Object/Complete/Test/  ← *_yolo.csv
    ...
```

**Run this once** before opening `FYP2_Data_Analysis_Nandana2026.ipynb`.  
Already-existing CSVs are skipped automatically.

In [ ]:
import cv2
import mediapipe as mp
from ultralytics import YOLO
import os
import csv
from pathlib import Path

BASE_DIR     = Path('/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/An_upper_limb_stroke_rehabilitation_exercise_video')
EXERCISE_DIR = BASE_DIR / 'Exercise'
YOLO_OUT_DIR = BASE_DIR / 'YOLO_CSV'
MP_OUT_DIR   = BASE_DIR / 'MediaPipe_CSV'


def extract_keypoints(video_path, yolo_target_dir, mp_target_dir, mp_pose_model, yolo_model):
    """Run MediaPipe and YOLOv8 on a single video; write one CSV each."""
    cap = cv2.VideoCapture(str(video_path))

    video_stem   = video_path.stem
    yolo_csv_path = yolo_target_dir / f'{video_stem}_yolo.csv'
    mp_csv_path   = mp_target_dir   / f'{video_stem}_mp.csv'

    # Skip if already extracted
    if yolo_csv_path.exists() and mp_csv_path.exists():
        print(f'  [SKIP] {video_path.name} — CSVs already exist')
        cap.release()
        return

    with open(yolo_csv_path, 'w', newline='') as y_file, \
         open(mp_csv_path,   'w', newline='') as m_file:

        y_writer = csv.writer(y_file)
        m_writer = csv.writer(m_file)

        y_header = ['frame'] + [f'kp{i}_{c}' for i in range(17) for c in ('x', 'y', 'conf')]
        m_header = ['frame'] + [f'lm{i}_{c}' for i in range(33) for c in ('x', 'y', 'z', 'vis')]
        y_writer.writerow(y_header)
        m_writer.writerow(m_header)

        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # ── YOLOv8 ────────────────────────────────────────────────────────
            results = yolo_model(frame, verbose=False)
            y_row = [frame_idx]
            if results and results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
                kpts = results[0].keypoints.data[0].cpu().numpy()
                for kp in kpts:
                    y_row.extend([kp[0], kp[1], kp[2]])
            else:
                y_row.extend([0] * 17 * 3)
            y_writer.writerow(y_row)

            # ── MediaPipe ─────────────────────────────────────────────────────
            frame_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_results = mp_pose_model.process(frame_rgb)
            m_row = [frame_idx]
            if mp_results.pose_landmarks:
                for lm in mp_results.pose_landmarks.landmark:
                    m_row.extend([lm.x, lm.y, lm.z, lm.visibility])
            else:
                m_row.extend([0] * 33 * 4)
            m_writer.writerow(m_row)

            frame_idx += 1

    cap.release()
    print(f'  -> Saved: {yolo_csv_path.name} & {mp_csv_path.name}')


def process_videos():
    mp_pose_ctx = mp.solutions.pose
    pose = mp_pose_ctx.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
    )
    print('Loading YOLOv8n-pose...')
    yolo_model = YOLO('yolov8n-pose.pt')

    videos = sorted(EXERCISE_DIR.rglob('*.mp4'))
    videos = [v for v in videos if 'DASHBOARD' not in v.name]
    print(f'Found {len(videos)} videos. Starting extraction...\n')

    for video_path in videos:
        relative_subfolder = video_path.parent.relative_to(EXERCISE_DIR)
        yolo_target_dir = YOLO_OUT_DIR / relative_subfolder
        mp_target_dir   = MP_OUT_DIR   / relative_subfolder
        yolo_target_dir.mkdir(parents=True, exist_ok=True)
        mp_target_dir.mkdir(parents=True, exist_ok=True)

        print(f'Processing: {video_path.name}  [{relative_subfolder}]')
        extract_keypoints(video_path, yolo_target_dir, mp_target_dir, pose, yolo_model)

    pose.close()
    print('\n✅ All videos processed.')


process_videos()